In [ ]:

from Wordle_Entropy_dictionarysortedalphabetically import Wordle
import numpy as np
from tqdm import tqdm
from matplotlib import pyplot as plt
import time


def play_wordle(alphabet, data_set_words, data_set_answers, sol, length=5, attempts=6, mode="hard"):
    """
    alphabet          - set of symbols allowed
    data_set_words    - set of valid guesses L
    data_set_answers  - set of solutions S
    sol               - the solution to be played
    length            - length of the word
    attempts          - maximum number of allowed attempts
    mode              - difficulty ("normal" or "hard")
    """
    if sol not in data_set_answers:
        print("That is not a valid solution")
        return None
    if length < 1 or attempts < 1:
        print("The length of the words and the number of maximum attempts must be Natural numbers.")
        return None

    wordle = Wordle(alphabet, data_set_words, sol, length, attempts)
    wordle.all_patterns
    print("Welcome to Wordle!")
    print('Remember that you must enter a valid {} letters word.\n'.format(length))

    while wordle.continue_playing:
        y = input("Type your guess: ")
        if wordle.valid_word(wordle.language, y) == False:
            print('This is not a valid input.')
            print('You must enter a valid {} letters word.'.format(length))
            continue
        wordle.word_list(y)
        if wordle.you_win == True:
            print("You guessed the word!")
            return None
        wordle.pattern_print(y, sol)
        print("You've got {} attempts left. \n".format(wordle.Num_attempts - len(wordle.guesses)))
        if mode == "hard":
            wordle.classification_words
            wordle.partition

    if wordle.you_win == False:
        print("You failed to solve the puzzle. Good luck next time!")
    return None


def random_wordle(alphabet, data_set_words, data_set_answers, sol, length=5, attempts=6, mode="hard"):
    """
    Same as play_wordle, but guesses are chosen at random from data_set_words
    instead of being typed in by a human player.
    """
    if sol not in data_set_answers:
        print("That is not a valid solution")
        return None
    if length < 1 or attempts < 1:
        print("The length of the words and the number of maximum attempts must be Natural numbers.")
        return None

    wordle = Wordle(alphabet, data_set_words, sol, length, attempts)
    wordle.all_patterns
    print("Welcome to Wordle!")
    print('Remember that you must enter a valid {} letters word.\n'.format(length))

    while wordle.continue_playing:
        y = data_set_words[np.random.randint(len(data_set_words))]
        if wordle.valid_word(wordle.language, y) == False:
            print('This is not a valid input.')
            print('You must enter a valid {} letters word.'.format(length))
            continue
        wordle.word_list(y)
        if wordle.you_win == True:
            print("You guessed the word!")
            return None
        wordle.pattern_print(y, sol)
        print("You've got {} attempts left. \n".format(wordle.Num_attempts - len(wordle.guesses)))
        if mode == "hard":
            wordle.classification_words
            wordle.partition

    if wordle.you_win == False:
        print("You failed to solve the puzzle. Good luck next time!")
    return None


def top_10_init(alphabet, data_set_words, length=5):
    # compute the top 10 initial guesses (based on entropy) in the input set
    wordle = Wordle(alphabet, data_set_words, data_set_words[0], length)
    wordle.all_patterns
    wordle.best_opening_guess
    return None


def wordle_solver(alphabet, data_set_words, sol, openers, length=5, attempts=20, plot="Yes"):
    """
    Run the entropy-based algorithm in hard mode.

    alphabet         - set of symbols allowed
    data_set_words   - set of valid guesses L
    sol              - set of games (solutions) to be played
    openers          - list of words to use as initial guesses / score
    length           - length of the word
    attempts         - maximum number of allowed attempts
    plot             - "Yes"/"No", whether to generate score histograms
    """
    if length < 1 or attempts < 1:
        print("The length of the words and the number of maximum attempts must be Natural numbers.")
        return None

    scores = np.zeros(len(openers))
    wordle = Wordle(alphabet, data_set_words, data_set_words[0], length, attempts)
    wordle.all_patterns

    for opener in openers:
        print(opener)
        wins = 0
        Total_attempts = 0
        best_initial_guess = opener
        list_number_attempts = []
        start = time.time()

        for solution in tqdm(sol):
            attempt = 1
            Total_attempts += 1
            wordle.new_game(solution, data_set_words)
            while wordle.continue_playing:
                wordle.bits_uncertainty
                if len(wordle.language) == len(data_set_words):
                    y = best_initial_guess
                wordle.word_list(y)
                if wordle.you_win == True:
                    if attempt <= 6:
                        wins += 1
                    break
                wordle.pattern(y, solution)
                wordle.classification_words
                wordle.partition
                Total_attempts += 1
                attempt += 1
                y = wordle.max_entropy_guess
            list_number_attempts.append(attempt)

        end = time.time()
        print("\nThe algorithm needed {} seconds to play all games in the list sol.".format(end - start))
        average = wins * 100 / len(sol)
        print("The algorithm won {} games out of {} ({:.5}%).".format(wins, len(sol), average))
        average_guesses = Total_attempts / len(sol)
        scores[openers.index(opener)] = average_guesses

        attemps_eachtype = []
        for i in range(1, 11):
            attemps_eachtype.append(list_number_attempts.count(i))
        print(attemps_eachtype)
        print("The average number of guesses is {:.5}.".format(average_guesses))

        if plot == "Yes":
            x = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
            y_vals = attemps_eachtype
            plt.bar(x, y_vals, align='center')
            plt.title('Starter: {}. Score = {:.4}'.format(best_initial_guess, average_guesses))
            plt.xlabel('Number of attempts')
            plt.ylabel('Frequency')
            plt.show()

    print(scores)
    return None


def plot_scores(openers, scores):
    # plot words in `openers` (ranked by entropy) against their average scores
    x = [i for i in range(1, len(openers) + 1)]
    plt.stem(x, scores)
    plt.ylim((min(scores) - 0.15, max(scores) + 0.025))
    plt.xlabel('Ranked words based on entropy')
    plt.ylabel('Average score over all possible games')
    plt.show()
    return None


def main():
    # Load the set of valid guesses, the set of solutions and the alphabet
    data_set_words = np.genfromtxt('valid-wordle-words.txt', delimiter='\n', dtype=str)
    data_set_answers = np.genfromtxt('wordle-nyt-answers-alphabetical.txt', delimiter='\n', dtype=str)
    alphabet = np.genfromtxt('english_alphabet.txt', delimiter='\n', dtype=str)

    # --- Examples (uncomment to run) ---

    # Compute best initial guesses
    # top_10_init(alphabet, data_set_words, 5)     # all words
    # top_10_init(alphabet, data_set_answers, 5)   # list of solutions

    # Play wordle in normal mode with solution "zebra"
    # sol = "zebra"
    # play_wordle(alphabet, data_set_words, data_set_answers, sol, 5, 6, 'normal')

    # Play wordle in hard mode with solution "zebra"
    # sol = "zebra"
    # play_wordle(alphabet, data_set_words, data_set_answers, sol, 5, 6)

    # Play wordle in normal mode with a random solution
    # sol = np.random.choice(data_set_answers)
    # play_wordle(alphabet, data_set_words, data_set_answers, sol, 5, 6, 'normal')

    # Play wordle in hard mode with a random solution
    # sol = np.random.choice(data_set_answers)
    # play_wordle(alphabet, data_set_words, data_set_answers, sol, 5, 6, 'hard')

    # Run the solver on a single game: solution "zebra", opener "tales"
    # openers = ["tales"]
    # sol = ["zebra"]
    # wordle_solver(alphabet, data_set_words, sol, openers, length=5, attempts=20, plot="No")

    # Run the solver over every possible game with opener "tales"
    # openers = ["tales"]
    # wordle_solver(alphabet, data_set_words, data_set_answers, openers, length=5, attempts=20, plot="Yes")

    # Run the solver restricting guesses to the set of solutions, opener "slate"
    # openers = ["slate"]
    # wordle_solver(alphabet, data_set_answers, data_set_answers, openers, length=5, attempts=20, plot="Yes")

    # Run the solver over the top 50 initial guesses (based on entropy)
    # openers = ['tares', 'lares', 'rales', 'rates', 'teras', 'nares', 'soare', 'tales', 'reais', 'tears',
    #            'arles', 'tores', 'salet', 'aeros', 'dares', 'saner', 'reals', 'lears', 'lores', 'serai',
    #            'lanes', 'laers', 'pares', 'cares', 'tires', 'saine', 'seral', 'mares', 'reans', 'aloes',
    #            'sared', 'roles', 'teals', 'aures', 'earls', 'taels', 'raise', 'tries', 'rotes', 'sorel',
    #            'leats', 'nears', 'toeas', 'strae', 'rones', 'nates', 'earns', 'taser', 'toles', 'dales']
    # wordle_solver(alphabet, data_set_words, data_set_answers, openers, length=5, attempts=20, plot="Yes")

    # 8-letter words
    # data_set_words_8 = np.genfromtxt('dict_8letterword.txt', delimiter='\n', dtype=str)
    # data_set_answers_8 = np.genfromtxt('dict_8letterword_sol.txt', delimiter='\n', dtype=str)
    # openers = ['pantries', 'calories', 'ratlines', 'pertains', 'latrines', 'serotina', 'realties',
    #            'canotier', 'marlites', 'toenails']
    # wordle_solver(alphabet, data_set_words_8, data_set_answers_8, openers, length=8, attempts=20, plot="Yes")

    # 10-letter words
    # data_set_words_10 = np.genfromtxt('dict_10letterwords.txt', delimiter='\n', dtype=str)
    # data_set_answers_10 = np.genfromtxt('dict_10letterwords_sol.txt', delimiter='\n', dtype=str)
    # openers = ['porcelains', 'centralise', 'contraries', 'polarities', 'moralities', 'tolerances',
    #            'categories', 'meliorates', 'penalities', 'centralism']
    # wordle_solver(alphabet, data_set_words_10, data_set_answers_10, openers, length=10, attempts=20, plot="Yes")

    # 14-letter words
    # data_set_words_14 = np.genfromtxt('dict_14letterword.txt', delimiter='\n', dtype=str)
    # data_set_answers_14 = np.genfromtxt('dict_14letterword_sol.txt', delimiter='\n', dtype=str)
    # openers = ['abolitionising']
    # wordle_solver(alphabet, data_set_words_14, data_set_answers_14, openers, length=14, attempts=20, plot="Yes")


if __name__ == "__main__":
    main()
